# 🌾 Sri Lanka Paddy Price Prediction AI

This notebook handles the enhanced model development pipeline, incorporating the **Maha** and **Yala** agricultural seasons.

**Objectives:**
1. **Validation**: Quality check the cleaned dataset.
2. **EDA**: Understand price behavior during Maha/Yala seasons.
3. **Engineering**: Create powerful time-series features.
4. **Modeling**: Train an XGBoost Regressor to forecast prices.
5. **Evaluation**: Measure performance with MAE, RMSE, and MAPE.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

plt.style.use('fivethirtyeight')
print("Setup complete.")

## 1. Data Loading & Validation
Loading the enhanced `dataset.jsonl` with seasonal markers.

In [ ]:
# Load data
df = pd.read_json('data/dataset.jsonl', lines=True)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['district', 'date'])

# Validation Check
print("--- Data Validation Summary ---")
print(f"Total records: {len(df)}")
print(f"Date Range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Missing values in any column: {df.isnull().sum().sum()}")
print(f"Unique Districts: {df['district'].nunique()}")
df.head()

## 2. Exploratory Data Analysis (EDA)
Analyzing price trends by season (Maha vs. Yala).

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='month_num', y='samba_avg_price')
plt.title('Distribution of Samba Avg Price by Month')
plt.xlabel('Month')
plt.ylabel('Price (LKR)')
plt.show()

plt.figure(figsize=(8, 5))
season_avg = df.groupby(['is_maha', 'is_yala'])['samba_avg_price'].mean().reset_index()
season_labels = ['Yala Season', 'Maha Season']
plt.bar(season_labels, [season_avg.iloc[0]['samba_avg_price'], season_avg.iloc[1]['samba_avg_price']], color=['#ff9999','#66b3ff'])
plt.title('Average Samba Price by Season')
plt.ylabel('Avg Price')
plt.show()

## 3. High-Performance Feature Engineering

In [ ]:
def create_advanced_features(df):
    df = df.copy()
    
    target = 'samba_avg_price'
    
    # Price Lags (Short & Long term)
    for lag in [1, 7, 30]:
        df[f'{target}_lag_{lag}'] = df.groupby('district')[target].shift(lag)
    
    # Rolling Mean & Standard Deviation (Volatility)
    for window in [7, 30]:
        df[f'{target}_roll_avg_{window}'] = df.groupby('district')[target].transform(lambda x: x.shift(1).rolling(window=window).mean())
        df[f'{target}_roll_std_{window}'] = df.groupby('district')[target].transform(lambda x: x.shift(1).rolling(window=window).std())
        
    # Rainfall trend
    df['rainfall_roll_7'] = df.groupby('district')['rainfall_mm'].transform(lambda x: x.shift(1).rolling(window=7).mean())
    
    return df.dropna()

df_final = create_advanced_features(df)
print(f"Final dataset ready with {df_final.shape[1]} features and {len(df_final)} rows.")

## 4. Model Development (XGBoost)

In [ ]:
# Time-Series Split
train = df_final[df_final['date'] < '2025-01-01']
test = df_final[df_final['date'] >= '2025-01-01']

features = [
    'month_num', 'is_maha', 'is_yala', 
    'samba_avg_price_lag_1', 'samba_avg_price_lag_7', 'samba_avg_price_lag_30',
    'samba_avg_price_roll_avg_7', 'samba_avg_price_roll_avg_30',
    'samba_avg_price_roll_std_7', 'samba_avg_price_roll_std_30',
    'rainfall_mm', 'temperature_c', 'rainfall_roll_7',
    'diesel_price', 'petrol_price'
]
target = 'samba_avg_price'

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Training with XGBoost
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=6,
    early_stopping_rounds=50,
    objective='reg:squarederror',
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)

## 5. Model Evaluation

In [ ]:
test = test.copy()
test['predictions'] = model.predict(X_test)

mae = mean_absolute_error(y_test, test['predictions'])
rmse = np.sqrt(mean_squared_error(y_test, test['predictions']))
mape = np.mean(np.abs((y_test - test['predictions']) / (y_test + 1e-9))) * 100

print(f"METRICS Summary:")
print(f"Mean Absolute Error: {mae:.2f} LKR")
print(f"Root Mean Squared Error: {rmse:.2f} LKR")
print(f"Mean Absolute Percentage Error: {mape:.2f}%")

# Visualize specific district performance
dist = 'Anuradhapura'
dist_test = test[test['district'] == dist]
plt.figure(figsize=(14, 7))
plt.plot(dist_test['date'], dist_test[target], label='Actual', alpha=0.6)
plt.plot(dist_test['date'], dist_test['predictions'], label='Predicted', alpha=0.8)
plt.title(f'Paddy Price Prediction Performance in {dist}')
plt.legend()
plt.show()

In [ ]:
# Feature Importance Breakdown
importance = pd.DataFrame(data=model.feature_importances_, index=model.feature_names_in_, columns=['importance'])
importance.sort_values('importance').plot(kind='barh', figsize=(10, 8), color='teal')
plt.title('Driving Factors for Paddy Prices (XGBoost Importance)')
plt.show()